# Pipeline completo de forecasting paso a paso

Este notebook recorre el pipeline entero del proyecto con un panel sintetico pequeno para que se entienda la logica sin depender de una descarga pesada ni de una ejecucion larga.

## Que vas a ver

- como es el panel de entrada;
- como se generan las features y el target;
- como se construyen los folds temporales;
- como entrenan el baseline y el modelo boosting;
- como se traduce el forecast a una decision de inventario;
- como se ejecuta el pipeline end-to-end con las funciones reales del repo.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
warnings.filterwarnings("ignore")

from retail_forecasting.config import DatasetConfig, ReportingConfig, Settings
from retail_forecasting.drift import label_stockout_regime
from retail_forecasting.evaluation.metrics import summarize_costs, summarize_predictions
from retail_forecasting.features.engineering import build_supervised_frame
from retail_forecasting.forecasting.backtesting import build_walk_forward_folds
from retail_forecasting.forecasting.pipeline import run_experiment_from_frame
from retail_forecasting.inventory.newsvendor import attach_inventory_costs, choose_order_quantity
from retail_forecasting.models.boosting import AutoBoostingModel
from retail_forecasting.models.naive import SeasonalNaiveModel
from retail_forecasting.utils.io import quantile_column_name
import retail_forecasting.models.boosting as boosting_module

# Workaround para macOS cuando LightGBM esta instalado pero no carga por falta de libomp.
# Asi el notebook cae al backend de scikit-learn y sigue siendo ejecutable.
boosting_module._lightgbm_available = lambda: False


## 1. Construimos un panel pequeno y entendible

En el proyecto real el panel sale de `FreshRetailNet-50K`, pero para entender la mecanica es mejor empezar con pocas series y pocos dias. El formato del panel sintetico imita el del pipeline real: una fila por `series_id` y fecha.


In [ ]:
def make_synthetic_panel(num_series: int = 3, num_days: int = 90) -> pd.DataFrame:
    dates = pd.date_range("2025-01-01", periods=num_days, freq="D")
    rows = []

    for store_id in range(1, num_series + 1):
        product_id = 100 + store_id
        for index, date in enumerate(dates):
            base = 10 + store_id
            seasonal = 2.0 * np.sin(2 * np.pi * index / 7)
            demand = max(base + seasonal + (index % 5) * 0.5, 0.1)
            rows.append(
                {
                    "city_id": 1,
                    "store_id": store_id,
                    "management_group_id": 1,
                    "first_category_id": 1,
                    "second_category_id": 1,
                    "third_category_id": store_id,
                    "product_id": product_id,
                    "date": date,
                    "observed_demand": demand,
                    "stockout_hours": float(index % 3),
                    "discount": 1.0 - 0.05 * (index % 2),
                    "holiday_flag": int(date.dayofweek == 6),
                    "activity_flag": int(index % 10 == 0),
                    "precpt": float(index % 4),
                    "avg_temperature": 15.0 + store_id,
                    "avg_humidity": 50.0 + index % 10,
                    "avg_wind_level": 3.0 + (index % 3),
                    "series_id": f"{store_id}_{product_id}",
                }
            )

    return pd.DataFrame(rows)


panel = make_synthetic_panel(num_series=3, num_days=90)
display(panel.head())
display(panel.groupby("series_id")["date"].agg(["min", "max", "count"]))


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,date,observed_demand,stockout_hours,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level,series_id
0,1,1,1,1,1,1,101,2025-01-01,11.000000,0.0,1.00,0,1,0.0,16.0,50.0,3.0,1_101
1,1,1,1,1,1,1,101,2025-01-02,13.063663,1.0,0.95,0,0,1.0,16.0,51.0,4.0,1_101
2,1,1,1,1,1,1,101,2025-01-03,13.949856,2.0,1.00,0,0,2.0,16.0,52.0,5.0,1_101
3,1,1,1,1,1,1,101,2025-01-04,13.367767,0.0,0.95,0,0,3.0,16.0,53.0,3.0,1_101
4,1,1,1,1,1,1,101,2025-01-05,12.132233,1.0,1.00,1,0,0.0,16.0,54.0,4.0,1_101


,min,max,count
series_id,,,
1_101,2025-01-01,2025-03-31,90
2_102,2025-01-01,2025-03-31,90
3_103,2025-01-01,2025-03-31,90


## 2. Del panel al frame supervisado

Aqui pasan dos cosas importantes:

1. etiquetamos un regimen de stockout para poder segmentar luego el analisis;
2. construimos las features y el target de demanda acumulada a `h` dias.

El target del proyecto no es la demanda de mañana, sino la demanda acumulada durante el horizonte de lead time.


In [3]:
settings = Settings()
settings.dataset = DatasetConfig(top_n_series=3, min_history_days=70, horizon=7)

prepared_panel = label_stockout_regime(panel)
supervised_frame, feature_columns = build_supervised_frame(
    panel=prepared_panel,
    feature_config=settings.features,
    horizon=settings.dataset.horizon,
)

print(f"Filas del panel preparado: {len(prepared_panel)}")
print(f"Filas del frame supervisado: {len(supervised_frame)}")
print(f"Numero de features: {len(feature_columns)}")
print("Primeras 12 features:", feature_columns[:12])

columns_to_show = [
    "series_id",
    "date",
    "observed_demand",
    "stockout_regime",
    "demand_lag_1",
    "demand_lag_7",
    "demand_roll_mean_7",
    "discount_lag_1",
    "stockout_lag_1",
    "target_lead_time_demand",
]
display(supervised_frame.loc[:, columns_to_show].head(10))


Filas del panel preparado: 270
Filas del frame supervisado: 168
Numero de features: 50
Primeras 12 features: ['day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend', 'holiday_flag', 'activity_flag', 'demand_lag_1', 'demand_lag_7', 'demand_lag_14', 'demand_lag_28', 'demand_roll_mean_7']


,series_id,date,observed_demand,stockout_regime,demand_lag_1,demand_lag_7,demand_roll_mean_7,discount_lag_1,stockout_lag_1,target_lead_time_demand
0,1_101,2025-01-29,12.500000,high_stockout,10.436337,11.500000,11.928571,0.95,0.0,85.5
1,1_101,2025-01-30,14.563663,high_stockout,12.500000,13.563663,12.071429,1.00,1.0,84.0
2,1_101,2025-01-31,12.949856,low_stockout,14.563663,14.449856,12.214286,0.95,2.0,82.5
3,1_101,2025-02-01,12.367767,high_stockout,12.949856,13.867767,12.000000,1.00,0.0,83.5
4,1_101,2025-02-02,11.132233,high_stockout,12.367767,10.132233,11.785714,0.95,1.0,84.5
5,1_101,2025-02-03,10.550144,low_stockout,11.132233,9.550144,11.928571,1.00,2.0,85.5
6,1_101,2025-02-04,11.436337,high_stockout,10.550144,10.436337,12.071429,0.95,0.0,84.0
7,1_101,2025-02-05,11.000000,high_stockout,11.436337,12.500000,12.214286,1.00,1.0,82.5
8,1_101,2025-02-06,13.063663,low_stockout,11.000000,14.563663,12.000000,0.95,2.0,83.5
9,1_101,2025-02-07,13.949856,high_stockout,13.063663,12.949856,11.785714,1.00,0.0,84.5


## 3. Miramos una fila concreta para entender el target

Este bloque reproduce la intuicion central del pipeline:

- `demand_lag_1` debe venir del pasado inmediato;
- `target_lead_time_demand` debe ser la suma de la demanda futura durante `h` dias.


In [5]:
example_row = supervised_frame.iloc[0]
series_history = (
    prepared_panel.loc[prepared_panel["series_id"] == example_row["series_id"]]
    .sort_values("date")
    .reset_index(drop=True)
)
source_index = series_history.index[series_history["date"] == example_row["date"]][0]

expected_lag_1 = series_history.loc[source_index - 1, "observed_demand"]
expected_target = series_history.loc[
    source_index : source_index + settings.dataset.horizon - 1,
    "observed_demand",
].sum()

inspection_window = series_history.loc[
    max(0, source_index - 3) : source_index + settings.dataset.horizon + 1,
    ["date", "observed_demand", "stockout_hours", "discount"],
].copy()

print("Serie:", example_row["series_id"])
print("Fecha de decision:", pd.Timestamp(example_row["date"]).date())
print("demand_lag_1 en el frame:", round(float(example_row["demand_lag_1"]), 4))
print("demand_lag_1 esperado:", round(float(expected_lag_1), 4))
print("target_lead_time_demand en el frame:", round(float(example_row["target_lead_time_demand"]), 4))
print("target esperado:", round(float(expected_target), 4))
display(inspection_window)


Serie: 1_101
Fecha de decision: 2025-01-29
demand_lag_1 en el frame: 10.4363
demand_lag_1 esperado: 10.4363
target_lead_time_demand en el frame: 85.5
target esperado: 85.5


,date,observed_demand,stockout_hours,discount
25,2025-01-26,10.132233,1.0,0.95
26,2025-01-27,9.550144,2.0,1.00
27,2025-01-28,10.436337,0.0,0.95
28,2025-01-29,12.500000,1.0,1.00
29,2025-01-30,14.563663,2.0,0.95
30,2025-01-31,12.949856,0.0,1.00
31,2025-02-01,12.367767,1.0,0.95
32,2025-02-02,11.132233,2.0,1.00
33,2025-02-03,10.550144,0.0,0.95
34,2025-02-04,11.436337,1.0,1.00


## 4. Construimos los folds walk-forward

La validacion no es aleatoria. Cada fold respeta el tiempo: se entrena con pasado y se valida en el siguiente bloque temporal.


In [6]:
folds = build_walk_forward_folds(
    panel=prepared_panel,
    validation_config=settings.validation,
    horizon=settings.dataset.horizon,
)

folds_df = pd.DataFrame(
    {
        "fold_id": [fold.fold_id for fold in folds],
        "train_end_date": [fold.train_end_date for fold in folds],
        "validation_start_date": [fold.validation_start_date for fold in folds],
        "validation_end_date": [fold.validation_end_date for fold in folds],
    }
)
display(folds_df)


,fold_id,train_end_date,validation_start_date,validation_end_date
0,0,2025-02-19,2025-02-26,2025-03-04
1,1,2025-02-26,2025-03-05,2025-03-11
2,2,2025-03-05,2025-03-12,2025-03-18


## 5. Entrenamos un fold a mano

Antes de lanzar el pipeline completo, aqui hacemos lo mismo a mano sobre el primer fold. Esto ayuda a entender que hace realmente `run_experiment_from_frame()` por dentro.


In [7]:
first_fold = folds[0]
train_mask = supervised_frame["date"] <= first_fold.train_end_date
validation_mask = (
    (supervised_frame["date"] >= first_fold.validation_start_date)
    & (supervised_frame["date"] <= first_fold.validation_end_date)
)

train_frame = supervised_frame.loc[train_mask].copy()
validation_frame = supervised_frame.loc[validation_mask].copy()

print(f"Filas train fold 0: {len(train_frame)}")
print(f"Filas validation fold 0: {len(validation_frame)}")

baseline_model = SeasonalNaiveModel(
    seasonal_period=settings.models.seasonal_period,
    horizon=settings.dataset.horizon,
).fit(prepared_panel)

boosting_model = AutoBoostingModel(
    quantiles=settings.models.quantiles,
    random_seed=settings.project.random_seed,
    n_estimators=settings.models.n_estimators,
    learning_rate=settings.models.learning_rate,
    max_depth=settings.models.max_depth,
)
boosting_model.fit(
    train_frame.loc[:, feature_columns],
    train_frame["target_lead_time_demand"],
)

baseline_eval = validation_frame.loc[
    :, ["date", "series_id", "target_lead_time_demand", "stockout_hours", "stockout_regime"]
].copy()
baseline_eval["y_true"] = baseline_eval["target_lead_time_demand"]
baseline_eval["y_pred"] = baseline_model.predict(validation_frame)
baseline_eval["model_name"] = baseline_model.model_name
baseline_eval["backend_name"] = "heuristic"
baseline_eval["fold_id"] = first_fold.fold_id
baseline_eval["order_quantity"] = choose_order_quantity(
    predictions=baseline_eval,
    inventory_config=settings.inventory,
    quantile_columns=[],
    quantile_levels=[],
)
baseline_eval = attach_inventory_costs(baseline_eval, settings.inventory)

boost_eval = validation_frame.loc[
    :, ["date", "series_id", "target_lead_time_demand", "stockout_hours", "stockout_regime"]
].copy()
boost_eval["y_true"] = boost_eval["target_lead_time_demand"]
boost_eval["y_pred"] = boosting_model.predict(validation_frame.loc[:, feature_columns])
boost_eval["model_name"] = settings.models.point_model
boost_eval["backend_name"] = boosting_model.backend_name
boost_eval["fold_id"] = first_fold.fold_id

quantile_columns = []
quantile_predictions = boosting_model.predict_quantiles(validation_frame.loc[:, feature_columns])
for quantile in settings.models.quantiles:
    column = quantile_column_name(quantile)
    boost_eval[column] = quantile_predictions[column]
    quantile_columns.append(column)

boost_eval["order_quantity"] = choose_order_quantity(
    predictions=boost_eval,
    inventory_config=settings.inventory,
    quantile_columns=quantile_columns,
    quantile_levels=settings.models.quantiles,
)
boost_eval = attach_inventory_costs(boost_eval, settings.inventory)

predictions_fold_0 = pd.concat([baseline_eval, boost_eval], ignore_index=True)
metrics_summary, fold_metrics = summarize_predictions(predictions_fold_0)
cost_summary = summarize_costs(predictions_fold_0)

display(predictions_fold_0.head())
display(metrics_summary)
display(cost_summary)


Filas train fold 0: 66
Filas validation fold 0: 21


,date,series_id,target_lead_time_demand,stockout_hours,stockout_regime,y_true,y_pred,model_name,backend_name,fold_id,order_quantity,overstock_units,stockout_units,overstock_cost,stockout_cost,total_cost,q_0_1,q_0_5,q_0_9
0,2025-02-26,1_101,83.5,2.0,high_stockout,83.5,84.0,seasonal_naive,heuristic,0,84.0,0.5,0.0,0.5,0.0,0.5,NaN,NaN,NaN
1,2025-02-27,1_101,84.5,0.0,low_stockout,84.5,82.5,seasonal_naive,heuristic,0,82.5,0.0,2.0,0.0,8.0,8.0,NaN,NaN,NaN
2,2025-02-28,1_101,85.5,1.0,high_stockout,85.5,83.5,seasonal_naive,heuristic,0,83.5,0.0,2.0,0.0,8.0,8.0,NaN,NaN,NaN
3,2025-03-01,1_101,84.0,2.0,high_stockout,84.0,84.5,seasonal_naive,heuristic,0,84.5,0.5,0.0,0.5,0.0,0.5,NaN,NaN,NaN
4,2025-03-02,1_101,82.5,0.0,low_stockout,82.5,85.5,seasonal_naive,heuristic,0,85.5,3.0,0.0,3.0,0.0,3.0,NaN,NaN,NaN


,model_name,backend_name,observations,mae,rmse,pinball_q_0_1,pinball_q_0_5,pinball_q_0_9,coverage_q_0_1_q_0_9
0,auto_boosting,xgboost,21,0.005217,0.009032,0.261537,0.106327,0.354426,0.619048
1,seasonal_naive,heuristic,21,1.500000,1.762709,NaN,NaN,NaN,NaN


,model_name,backend_name,observations,mean_order_quantity,total_overstock_units,total_stockout_units,total_overstock_cost,total_stockout_cost,total_cost,mean_cost
0,auto_boosting,xgboost,21,93.693744,56.63956,0.070947,56.63956,0.283786,56.923347,2.710636
1,seasonal_naive,heuristic,21,90.785714,13.50000,18.000000,13.50000,72.000000,85.500000,4.071429


## 6. Ejecutamos el pipeline entero con una sola funcion

Una vez entendidos los bloques, este es el punto en el que el proyecto real junta todo: features, backtesting, modelos, decision de inventario y escritura de artefactos.


In [8]:
demo_settings = Settings()
demo_settings.dataset = DatasetConfig(top_n_series=3, min_history_days=70, horizon=7)
demo_settings.reporting = ReportingConfig(
    output_dir=PROJECT_ROOT / "reports",
    run_name="notebook_walkthrough",
    make_plots=False,
)

artifacts = run_experiment_from_frame(panel, demo_settings)

print("Run directory:", artifacts.run_directory)
display(artifacts.metrics_summary)
display(artifacts.cost_summary)
display(artifacts.predictions.head())


Run directory: /Users/artemmindlin/code/sandbox/retail-demand-forecasting-system/reports/notebook_walkthrough_20260401_180059


,model_name,backend_name,observations,mae,rmse,pinball_q_0_1,pinball_q_0_5,pinball_q_0_9,coverage_q_0_1_q_0_9
0,auto_boosting,xgboost,63,0.002565,0.005504,0.264725,0.080853,0.355198,0.746032
1,seasonal_naive,heuristic,63,1.547619,1.828999,NaN,NaN,NaN,NaN


,model_name,backend_name,observations,mean_order_quantity,total_overstock_units,total_stockout_units,total_overstock_cost,total_stockout_cost,total_cost,mean_cost
0,auto_boosting,xgboost,63,93.642531,168.682062,0.70262,168.682062,2.81048,171.492542,2.722104
1,seasonal_naive,heuristic,63,91.000000,49.500000,48.00000,49.500000,192.00000,241.500000,3.833333


,date,series_id,target_lead_time_demand,stockout_hours,stockout_regime,y_true,y_pred,model_name,backend_name,fold_id,order_quantity,overstock_units,stockout_units,overstock_cost,stockout_cost,total_cost,q_0_1,q_0_5,q_0_9
0,2025-02-26,1_101,83.5,2.0,high_stockout,83.5,84.0,seasonal_naive,heuristic,0,84.0,0.5,0.0,0.5,0.0,0.5,NaN,NaN,NaN
1,2025-02-27,1_101,84.5,0.0,low_stockout,84.5,82.5,seasonal_naive,heuristic,0,82.5,0.0,2.0,0.0,8.0,8.0,NaN,NaN,NaN
2,2025-02-28,1_101,85.5,1.0,high_stockout,85.5,83.5,seasonal_naive,heuristic,0,83.5,0.0,2.0,0.0,8.0,8.0,NaN,NaN,NaN
3,2025-03-01,1_101,84.0,2.0,high_stockout,84.0,84.5,seasonal_naive,heuristic,0,84.5,0.5,0.0,0.5,0.0,0.5,NaN,NaN,NaN
4,2025-03-02,1_101,82.5,0.0,low_stockout,82.5,85.5,seasonal_naive,heuristic,0,85.5,3.0,0.0,3.0,0.0,3.0,NaN,NaN,NaN


## 7. Como se conecta este notebook con los archivos del repo

- `src/retail_forecasting/data/fresh_retailnet.py`: carga el dataset real y prepara el panel.
- `src/retail_forecasting/features/engineering.py`: crea lags, rolling windows y el target `target_lead_time_demand`.
- `src/retail_forecasting/forecasting/backtesting.py`: define los folds walk-forward.
- `src/retail_forecasting/models/naive.py`: baseline seasonal naive.
- `src/retail_forecasting/models/boosting.py`: modelo boosting y cuantiles.
- `src/retail_forecasting/inventory/newsvendor.py`: pasa del forecast a `order_quantity` y calcula costes.
- `src/retail_forecasting/evaluation/metrics.py`: resume errores y costes.
- `src/retail_forecasting/forecasting/pipeline.py`: orquesta todo.
- `src/retail_forecasting/evaluation/reporting.py`: escribe `report.md`, CSVs y plots.

## Siguiente paso util

Cuando entiendas este notebook, cambia `panel = make_synthetic_panel(...)` por una llamada a `load_prepared_panel(...)` y veras exactamente el mismo pipeline sobre datos reales.
